# Customer Churn Analysis & Prediction

**Data Science Project — Customer Churn Analysis**

A comprehensive analysis of customer churn using the IBM Telco dataset. This notebook explores customer demographics, service usage patterns, and builds predictive models to identify customers at risk of churning.

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette(['#00e676', '#f44336'])
print('Libraries loaded successfully')

---
## 2. Data Loading

In [ ]:
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp/master/data/TELCO_CUSTOMER_CHURN_DATA.csv'
df = pd.read_csv(URL)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

---
## 3. Exploratory Data Analysis

### 3.1 Data Overview

In [ ]:
print('Dataset Info:')
print(f'  Shape: {df.shape}')
print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\nData Types:')
print(df.dtypes.value_counts())

In [ ]:
print('Missing Values:')
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print('  None')

### 3.2 Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Stayed', 'Churned'], churn_counts.values, color=['#00e676', '#f44336'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution (Count)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, f'{v}', ha='center', fontweight='bold', fontsize=12)

axes[1].pie(churn_pct.values, labels=['Stayed', 'Churned'], autopct='%1.1f%%',
            colors=['#00e676', '#f44336'], startangle=90, explode=(0, 0.05),
            textprops={'fontweight': 'bold'})
axes[1].set_title('Churn Distribution (Percentage)', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

print(f'Overall Churn Rate: {churn_pct["Yes"]:.2f}%')
print(f'This is a class-imbalanced problem (~27% churn rate)')

### 3.3 Demographic Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

gender_churn = pd.crosstab(df['gender'], df['Churn'], normalize='index') * 100
gender_churn.plot(kind='bar', ax=axes[0], color=['#00e676', '#f44336'], edgecolor='white', legend=False)
axes[0].set_title('Churn by Gender', fontweight='bold')
axes[0].set_ylabel('Percentage')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

partner_churn = pd.crosstab(df['Partner'], df['Churn'], normalize='index') * 100
partner_churn.plot(kind='bar', ax=axes[1], color=['#00e676', '#f44336'], edgecolor='white', legend=False)
axes[1].set_title('Churn by Partner Status', fontweight='bold')
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

dependents_churn = pd.crosstab(df['Dependents'], df['Churn'], normalize='index') * 100
dependents_churn.plot(kind='bar', ax=axes[2], color=['#00e676', '#f44336'], edgecolor='white', legend=False)
axes[2].set_title('Churn by Dependents', fontweight='bold')
axes[2].set_ylabel('Percentage')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()
print('Insight: Customers with partners and dependents churn less — they have more household dependency on the service.')

### 3.4 Tenure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[df['Churn'] == 'No']['tenure'].hist(ax=axes[0], bins=30, color='#00e676', alpha=0.7, edgecolor='white')
axes[0].set_title('Tenure Distribution - Stayed Customers', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')

df[df['Churn'] == 'Yes']['tenure'].hist(ax=axes[1], bins=30, color='#f44336', alpha=0.7, edgecolor='white')
axes[1].set_title('Tenure Distribution - Churned Customers', fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Avg Tenure (Stayed): {df[df["Churn"]=="No"]["tenure"].mean():.1f} months')
print(f'Avg Tenure (Churned): {df[df["Churn"]=="Yes"]["tenure"].mean():.1f} months')
print('\nKey Insight: Customers who churn have significantly lower tenure. The first 12 months are critical.')

### 3.5 Contract & Payment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=axes[0], color=['#00e676', '#f44336'], edgecolor='white')
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Percentage')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(['Stayed', 'Churned'])

payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.plot(kind='bar', ax=axes[1], color=['#00e676', '#f44336'], edgecolor='white')
axes[1].set_title('Churn Rate by Payment Method', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Percentage')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')
axes[1].legend(['Stayed', 'Churned'])

plt.tight_layout()
plt.show()

print('Key Insights:')
print('  - Month-to-month contracts have the highest churn rate (~43%)')
print('  - Two-year contracts have the lowest churn rate (~3%)')
print('  - Electronic check payment method correlates with higher churn')
print('  - Automatic payment methods (bank transfer, credit card) have lower churn')

### 3.6 Service Analysis

In [ ]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(service_cols):
    ax = axes[i // 3, i % 3]
    service_churn = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    service_churn.plot(kind='bar', ax=ax, color=['#00e676', '#f44336'], edgecolor='white', legend=False)
    ax.set_title(f'Churn by {col}', fontweight='bold')
    ax.set_ylabel('Percentage')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=10)

plt.tight_layout()
plt.show()
print('Key Insight: Customers without Online Security and Tech Support churn at much higher rates.')

### 3.7 Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes, ['MonthlyCharges', 'TotalCharges'], ['Monthly Charges', 'Total Charges']):
    for churn_status, color in [('No', '#00e676'), ('Yes', '#f44336')]:
        subset = df[df['Churn'] == churn_status][col]
        subset.hist(ax=ax, bins=30, alpha=0.6, color=color, label=f'{"Stayed" if churn_status=="No" else "Churned"}', edgecolor='white')
    ax.set_title(f'{title} Distribution by Churn Status', fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

print(f'Avg Monthly Charges (Stayed): ${df[df["Churn"]=="No"]["MonthlyCharges"].mean():.2f}')
print(f'Avg Monthly Charges (Churned): ${df[df["Churn"]=="Yes"]["MonthlyCharges"].mean():.2f}')
print(f'\nInsight: Higher monthly charges correlate with higher churn.')

### 3.8 Correlation Analysis

In [ ]:
df_encoded = df.copy()
df_encoded['Churn_Flag'] = (df_encoded['Churn'] == 'Yes').astype(int)
df_encoded['Gender_Flag'] = (df_encoded['gender'] == 'Male').astype(int)

numeric_df = df_encoded.select_dtypes(include=[np.number])
correlation = numeric_df.corr()['Churn_Flag'].sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#f44336' if v > 0 else '#00e676' for v in correlation.values]
correlation.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Feature Correlation with Churn', fontweight='bold', fontsize=14)
plt.xlabel('Features')
plt.ylabel('Correlation with Churn')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Top Churn Correlates (Positive):')
print(correlation.head(5).to_string())
print('\nTop Churn Correlates (Negative):')
print(correlation.tail(5).to_string())

---
## 4. Data Preprocessing

In [ ]:
df_clean = df.drop('customerID', axis=1)
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(df_clean['TotalCharges'].median())
df_clean['Gender'] = (df_clean['gender'] == 'Male').astype(int)
df_clean = df_clean.drop('gender', axis=1)

services = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for s in services:
    df_clean[f'{s}_Yes'] = (df_clean[s] == 'Yes').astype(int)
    df_clean = df_clean.drop(s, axis=1)

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for c in binary_cols:
    df_clean[c] = (df_clean[c] == 'Yes').astype(int)

cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
bool_cols = df_clean.select_dtypes(include=['bool']).columns
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

print(f'Final feature count: {df_clean.shape[1] - 1}')
print(f'Features: {[c for c in df_clean.columns if c != "Churn"]}')
df_clean.head()

---
## 5. Model Building

In [ ]:
X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_test[col] = scaler.transform(X_test[[col]])

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'Features:  {X_train.shape[1]}')

### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print('Logistic Regression Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_lr):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_lr):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_lr):.4f}')

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('Random Forest Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_rf):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_rf):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.4f}')

### 5.3 XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42, eval_metric='auc')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('XGBoost Results:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_xgb):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred_xgb):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred_xgb):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_test, y_proba_xgb):.4f}')

---
## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Logistic Regression': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_proba_lr)
    ],
    'Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf),
        roc_auc_score(y_test, y_proba_rf)
    ],
    'XGBoost': [
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb),
        roc_auc_score(y_test, y_proba_xgb)
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])

results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results.index))
width = 0.25
multiplier = 0

for model_name in results.columns:
    offset = width * multiplier
    bars = ax.bar(x + offset, results[model_name], width, label=model_name, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, results[model_name]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    multiplier += 1

ax.set_xticks(x + width)
ax.set_xticklabels(results.index)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 7. Business Insights & Recommendations

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'].values, color='#00e676', edgecolor='white')
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance')
plt.title('Top 15 Features for Churn Prediction', fontweight='bold', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print('='*60)
print('BUSINESS INSIGHTS')
print('='*60)
print('''
1. Contract Type is the strongest churn predictor
   → Encourage month-to-month customers to switch to annual contracts
     with incentives (discount, free month, etc.)

2. Tenure (months with company) is the second strongest
   → First 6 months are the highest-risk period
   → Implement a structured onboarding program with milestone check-ins

3. Customers without Online Security & Tech Support churn more
   → Bundle these services at a discount for high-risk customers
   → Educate customers about security features during onboarding

4. Electronic check payment method correlates with higher churn
   → Offer incentives for switching to automatic payment methods
   → Auto-pay discount of $5/month could reduce churn

5. Higher monthly charges increase churn probability
   → Implement loyalty discounts starting at 12-month milestone
   → Create tiered pricing with long-term commitment benefits

6. Fiber optic internet has higher churn than DSL
   → Likely due to price sensitivity or competitor offers
   → Monitor competitor pricing and offer retention-matching
''')

print('='*60)
print('ROI ESTIMATION')
print('='*60)
churn_rate = df['Churn'].value_counts(normalize=True)['Yes']
avg_revenue = df['MonthlyCharges'].mean()
total_customers = len(df)
preventable = int(total_customers * churn_rate * 0.6)
annual_savings = preventable * avg_revenue * 12
program_cost = total_customers * 20
net = annual_savings - program_cost
roi = (net / program_cost) * 100

print(f'''
  Total customers:            {total_customers:,}
  Current churn rate:         {churn_rate:.1%}
  At-risk customers/year:     ~{int(total_customers * churn_rate):,}
  Preventable with program:   {preventable:,} (60% retention rate)
  Annual revenue saved:       ${annual_savings:,.0f}
  Retention program cost:     ${program_cost:,.0f}
  Net annual impact:          ${net:,.0f}
  ROI:                        {roi:.1f}%
''')
print('='*60)